In [2]:
import numpy as np
import pandas as pd
from skforecast.utils import save_forecaster
from skforecast.utils import load_forecaster

In [16]:
# Exogenous features helpers
from features import set_holidays, cal_features, cyclic_features

In [3]:
import sys
import os
sys.path.append(os.pardir)

In [6]:
forecaster_loaded = load_forecaster('../model/forecaster_001.joblib', verbose=True, suppress_warnings=False)

/Users/maxwellgriffith/miniconda3/envs/loadcast/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ForecasterRecursive 
Estimator: LGBMRegressor 
Lags: [  1   2   3  20  21  22  23  24  25  26  27  28  29 146 167 170] 
Window features: None 
Window size: 170 
Series name: Demand 
Exogenous included: True 
Exogenous names: 
    Temperature, Holiday, week, day_of_week, hour, week_sin, week_cos, hour_sin,
    hour_cos, Temp_3D_Mean, Temp_2D_Max, Temp_2D_Min, Temp_1D_Min 
Categorical features: auto 
Transformer for y: None 
Transformer for exog: None 
Weight function included: False 
Differentiation order: None 
Drop NaN from series: False 
Training range: [Timestamp('2023-01-01 00:00:00'), Timestamp('2024-12-31 23:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <Hour> 
Estimator parameters: 
    {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0,
    'importance_type': 'split', 'learning_rate': 0.16546988609195726,
    'max_depth': 3, 'min_child_samples': 20, 'min_child_weight': 0.001,
    'min_split_gain': 0.0, 'n_estimators': 700, 'n_jobs'

- make regular predictions without retraining model using last_window
- This argument allows providing only the past values needed to create the autoregressive predictors ie lags
- When using the last_window argument, it is crucial to ensure that the length of last_window is sufficient to include the maximum lag (or custom predictor) used by the forecaster. For instance, if the forecaster employs lags 1, 24, and 48, last_window must include the most recent 48 values of the series

In [8]:
forecaster_loaded.last_window_

,Demand
date,
2024-12-24 22:00:00,30801.208
2024-12-24 23:00:00,31410.891
2024-12-25 00:00:00,31305.535
2024-12-25 01:00:00,30949.415
2024-12-25 02:00:00,30649.751
...,...
2024-12-31 19:00:00,32951.913
2024-12-31 20:00:00,32711.893
2024-12-31 21:00:00,32615.968


In [19]:
exo_vars = forecaster_loaded.exog_names_in_
exo_vars

['Temperature',
 'Holiday',
 'week',
 'day_of_week',
 'hour',
 'week_sin',
 'week_cos',
 'hour_sin',
 'hour_cos',
 'Temp_3D_Mean',
 'Temp_2D_Max',
 'Temp_2D_Min',
 'Temp_1D_Min']

In [9]:
#last window is the end of the validation split so we don't need to pull new load from gridstatus to tests the prod for now

In [10]:
df = (pd.read_csv("../data/raw/gsloadtemp_clean.csv")
        .drop_duplicates()
        .pipe(lambda df: df.set_index(pd.to_datetime(df["date"])))
        .drop(columns = ["date"])
     )
#this will throw an error if there are duplicates
df.index = df.index.tz_localize(None)
df.index.freq = 'h'

In [11]:
df.head()

,Demand,Temperature
date,,
2023-01-01 00:00:00,28771.933,6.90
2023-01-01 01:00:00,28488.282,7.70
2023-01-01 02:00:00,28073.965,6.55
2023-01-01 03:00:00,27756.863,5.80
2023-01-01 04:00:00,27271.343,5.65


In [12]:
TEST_START  = "2025-01-01 00:00:00"
TEST_END    = "2025-12-31 23:00:00"

In [14]:
data_test  = df.loc[TEST_START:, :].copy()

In [15]:
print(f"Test dates       : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")

Test dates       : 2025-01-01 00:00:00 --- 2025-12-31 23:00:00  (n=8760)


- years worth of data to test on
- lags 170
- exogenous variables
    - Temperature
    - Holiday
    - week
    - day_of_week
    - hour
    - week_sin
    - week_cos
    - hour_sin
    - hour_cos
    - Temp_3D_Mean
    - Temp_2D_Max
    - Temp_2D_Min
    - Temp_1D_Min 

In [20]:
data_test = (data_test
            .pipe(set_holidays)
            .pipe(cal_features)
            .pipe(cyclic_features)
            .assign(Holiday=lambda d: d["Holiday"].astype(int))
            .assign(Temp_3D_Mean=lambda d: d["Temperature"].rolling("3D", center = False).mean())
            .assign(Temp_2D_Max =lambda d: d["Temperature"].rolling("2D", center = False).max())
            .assign(Temp_2D_Min =lambda d: d["Temperature"].rolling("2D", center = False).min())
            .assign(Temp_1D_Min =lambda d: d["Temperature"].rolling("1D", center = False).min())
           )

In [21]:
data_test.columns

Index(['Demand', 'Temperature', 'Holiday', 'month', 'week', 'day_of_week',
       'hour', 'month_sin', 'month_cos', 'week_sin', 'week_cos', 'day_sin',
       'day_cos', 'hour_sin', 'hour_cos', 'Temp_3D_Mean', 'Temp_2D_Max',
       'Temp_2D_Min', 'Temp_1D_Min'],
      dtype='str')

In [22]:
data_test[exo_vars]

,Temperature,Holiday,week,day_of_week,hour,week_sin,week_cos,hour_sin,hour_cos,Temp_3D_Mean,Temp_2D_Max,Temp_2D_Min,Temp_1D_Min
date,,,,,,,,,,,,,
2025-01-01 00:00:00,0.70,1,1,2,0,0.120537,0.992709,0.000000,1.000000,0.700000,0.70,0.70,0.70
2025-01-01 01:00:00,0.10,0,1,2,1,0.120537,0.992709,0.258819,0.965926,0.400000,0.70,0.10,0.10
2025-01-01 02:00:00,0.15,0,1,2,2,0.120537,0.992709,0.500000,0.866025,0.316667,0.70,0.10,0.10
2025-01-01 03:00:00,0.20,0,1,2,3,0.120537,0.992709,0.707107,0.707107,0.287500,0.70,0.10,0.10
2025-01-01 04:00:00,0.05,0,1,2,4,0.120537,0.992709,0.866025,0.500000,0.240000,0.70,0.05,0.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 19:00:00,8.35,0,1,2,19,0.120537,0.992709,-0.965926,0.258819,-3.354861,8.35,-8.05,-1.20
2025-12-31 20:00:00,9.30,0,1,2,20,0.120537,0.992709,-0.866025,0.500000,-3.203472,9.30,-8.05,-1.20
2025-12-31 21:00:00,9.75,0,1,2,21,0.120537,0.992709,-0.707107,0.707107,-3.011111,9.75,-8.05,-1.20


In [24]:
forecaster_loaded.predict(
    steps = 24,
    exog = data_test[exo_vars]
)

2025-01-01 00:00:00    34266.067496
2025-01-01 01:00:00    34233.980226
2025-01-01 02:00:00    33924.012688
2025-01-01 03:00:00    33377.508707
2025-01-01 04:00:00    32431.177194
2025-01-01 05:00:00    31456.868182
2025-01-01 06:00:00    30908.842679
2025-01-01 07:00:00    30701.304699
2025-01-01 08:00:00    30585.089811
2025-01-01 09:00:00    30794.115410
2025-01-01 10:00:00    31296.782683
2025-01-01 11:00:00    32258.764408
2025-01-01 12:00:00    33559.258594
2025-01-01 13:00:00    34513.908839
2025-01-01 14:00:00    34644.441200
2025-01-01 15:00:00    34367.594721
2025-01-01 16:00:00    34137.029226
2025-01-01 17:00:00    33665.437853
2025-01-01 18:00:00    33200.678524
2025-01-01 19:00:00    32806.524986
2025-01-01 20:00:00    32447.158699
2025-01-01 21:00:00    32213.572052
2025-01-01 22:00:00    32567.169344
2025-01-01 23:00:00    33501.410006
Freq: h, Name: pred, dtype: float64

In [ ]:
#now we use "last window"